# 生成对抗网络（GAN）生成手写数字图

## 运行环境说明

**执行本 notebook 前必须先激活 venv：**
```powershell
.\venv\Scripts\activate
```

## 任务说明

- 数据集：MNIST 手写数字（0-9 共 10 类，28×28 灰度图）
- 模型：生成对抗网络 GAN
  - 生成器 G：将随机噪声 z 映射为一张 28×28 的图像
  - 判别器 D：判断输入图像是真实样本还是 G 生成的假样本
- 损失函数：`BCEWithLogitsLoss`（二元交叉熵）
- 优化器：`Adam`（G 和 D 各自一个）
- 训练：每轮 print 损失值，训练结束生成并显示若干张手写数字图


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms, datasets
import numpy as np
import matplotlib.pyplot as plt

print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

torch: 2.0.1+cu118
torchvision: 0.15.2+cu118
device: cuda


In [3]:
# ============ 超参数 ============
batch_size = 64
lr = 2e-4           # GAN 推荐学习率
num_epochs = 20     # 训练轮数
z_dim = 100         # 噪声向量维度
image_dim = 28 * 28 # MNIST 图像像素数
num_channels = 1    # 灰度图

torch.manual_seed(42)
np.random.seed(42)

In [4]:
# ============ 加载 MNIST 数据集 ============
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # 归一化到 [-1, 1]
])

mnist_dataset = datasets.MNIST(
    root='./data',
    train=True,
    transform=transform,
    download=True
)

dataloader = DataLoader(mnist_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
print(f'MNIST 训练集大小：{len(mnist_dataset)}，共 {len(dataloader)} 个 batch')

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9912422/9912422 [00:05<00:00, 1700166.76it/s]


Extracting ./data\MNIST\raw\train-images-idx3-ubyte.gz to ./data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28881/28881 [00:00<00:00, 72283.40it/s]


Extracting ./data\MNIST\raw\train-labels-idx1-ubyte.gz to ./data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 1648877/1648877 [00:01<00:00, 1427142.84it/s]


Extracting ./data\MNIST\raw\t10k-images-idx3-ubyte.gz to ./data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 4542/4542 [00:00<00:00, 497701.72it/s]

Extracting ./data\MNIST\raw\t10k-labels-idx1-ubyte.gz to ./data\MNIST\raw

MNIST 训练集大小：60000，共 937 个 batch


In [5]:
# ============ 生成器 Generator ============
class Generator(nn.Module):
    """将 z_dim 维噪声向量映射为 1×28×28 的图像"""
    def __init__(self, z_dim, image_dim):
        super(Generator, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2),
            nn.Linear(1024, image_dim),
            nn.Tanh()  # 输出范围 [-1, 1]，与 Normalize 对齐
        )
    
    def forward(self, z):
        return self.net(z)

G = Generator(z_dim, image_dim).to(device)
print(G)

Generator(
  (net): Sequential(
    (0): Linear(in_features=100, out_features=256, bias=True)
    (1): LeakyReLU(negative_slope=0.2)
    (2): Linear(in_features=256, out_features=512, bias=True)
    (3): LeakyReLU(negative_slope=0.2)
    (4): Linear(in_features=512, out_features=1024, bias=True)
    (5): LeakyReLU(negative_slope=0.2)
    (6): Linear(in_features=1024, out_features=784, bias=True)
    (7): Tanh()
  )
)


In [6]:
# ============ 判别器 Discriminator ============
class Discriminator(nn.Module):
    """输入 1×28×28 图像，输出真假概率（logits）"""
    def __init__(self, image_dim):
        super(Discriminator, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(image_dim, 512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(256, 1)  # 输出 logits，搭配 BCEWithLogitsLoss
        )
    
    def forward(self, x):
        return self.net(x)

D = Discriminator(image_dim).to(device)
print(D)

Discriminator(
  (net): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): LeakyReLU(negative_slope=0.2)
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=512, out_features=256, bias=True)
    (4): LeakyReLU(negative_slope=0.2)
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=256, out_features=1, bias=True)
  )
)


In [7]:
# ============ 损失函数 & 优化器 ============
criterion = nn.BCEWithLogitsLoss()

optimizer_G = optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
optimizer_D = optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))

# 标签
real_label = 1.0
fake_label = 0.0

In [ ]:
# ============ 训练循环 ============
import os

os.makedirs('./gan_output', exist_ok=True)

G_losses = []
D_losses = []

print('开始训练 GAN...')
print(f"{'Epoch':>6} | {'D_loss':>10} | {'G_loss':>10}")
print('-' * 35)

for epoch in range(1, num_epochs + 1):
    epoch_D_loss = 0.0
    epoch_G_loss = 0.0
    n_batches = 0
    
    for real_imgs, _ in dataloader:
        batch = real_imgs.size(0)
        # 展平为 (batch, 784)
        real_imgs = real_imgs.view(batch, -1).to(device)
        
        # ---------- 训练判别器 D ----------
        optimizer_D.zero_grad()
        
        # 真实样本
        label_real = torch.full((batch, 1), real_label, device=device)
        output_real = D(real_imgs)
        loss_D_real = criterion(output_real, label_real)
        
        # 假样本
        z = torch.randn(batch, z_dim, device=device)
        fake_imgs = G(z).detach()  # detach 避免梯度传到 G
        label_fake = torch.full((batch, 1), fake_label, device=device)
        output_fake = D(fake_imgs)
        loss_D_fake = criterion(output_fake, label_fake)
        
        loss_D = loss_D_real + loss_D_fake
        loss_D.backward()
        optimizer_D.step()
        
        # ---------- 训练生成器 G ----------
        optimizer_G.zero_grad()
        
        z = torch.randn(batch, z_dim, device=device)
        fake_imgs = G(z)
        # G 希望 D 把假样本判为真
        output = D(fake_imgs)
        loss_G = criterion(output, torch.full((batch, 1), real_label, device=device))
        loss_G.backward()
        optimizer_G.step()
        
        epoch_D_loss += loss_D.item()
        epoch_G_loss += loss_G.item()
        n_batches += 1
    
    avg_D = epoch_D_loss / n_batches
    avg_G = epoch_G_loss / n_batches
    G_losses.append(avg_G)
    D_losses.append(avg_D)
    
    print(f"{epoch:>6d} | {avg_D:>10.4f} | {avg_G:>10.4f}")
    
    # 每 5 轮保存一次生成的样图
    if epoch % 5 == 0 or epoch == num_epochs:
        G.eval()
        with torch.no_grad():
            test_z = torch.randn(16, z_dim, device=device)
            gen_imgs = G(test_z).view(-1, 1, 28, 28).cpu()
            gen_imgs = (gen_imgs + 1) / 2  # 反归一化到 [0, 1]
        G.train()
        
        fig, axes = plt.subplots(2, 8, figsize=(12, 3))
        for i, ax in enumerate(axes.flat):
            ax.imshow(gen_imgs[i, 0], cmap='gray')
            ax.axis('off')
        plt.suptitle(f'Epoch {epoch} 生成结果')
        plt.tight_layout()
        plt.savefig(f'./gan_output/epoch_{epoch:03d}.png', dpi=80)
        plt.show()

print('\n训练完成！')

开始训练 GAN...
 Epoch |     D_loss |     G_loss
-----------------------------------
     1 |     1.0593 |     1.3102
     2 |     0.8721 |     1.9481


In [ ]:
# ============ 保存模型 + 损失曲线 ============
torch.save(G.state_dict(), './gan_output/generator.pth')
torch.save(D.state_dict(), './gan_output/discriminator.pth')
print('模型已保存到 ./gan_output/')

plt.figure(figsize=(8, 4))
plt.plot(G_losses, label='G loss')
plt.plot(D_losses, label='D loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('GAN Training Loss')
plt.grid(True)
plt.savefig('./gan_output/loss_curve.png', dpi=80)
plt.show()

In [ ]:
# ============ 用训练好的 G 生成并显示 32 张手写数字 ============
G.eval()
with torch.no_grad():
    test_z = torch.randn(32, z_dim, device=device)
    gen_imgs = G(test_z).view(-1, 1, 28, 28).cpu()
    gen_imgs = (gen_imgs + 1) / 2  # 反归一化到 [0, 1]

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(gen_imgs[i, 0], cmap='gray')
    ax.axis('off')
plt.suptitle('GAN 生成的手写数字')
plt.tight_layout()
plt.show()